# 6.4 多输入多输出通道

真实图像通常有多个通道，例如 RGB 图像有 3 个颜色通道。卷积层也可以输出多个通道，每个输出通道对应一个不同的卷积核组。


In [1]:
import torch


In [2]:
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y


## 多输入通道

对每个输入通道分别做二维互相关，再把所有通道的结果相加。


In [3]:
def corr2d_multi_in(X, K):
    return sum(corr2d(x, k) for x, k in zip(X, K))


X = torch.tensor([[[0.0, 1.0, 2.0],
                   [3.0, 4.0, 5.0],
                   [6.0, 7.0, 8.0]],
                  [[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0],
                   [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0],
                   [2.0, 3.0]],
                  [[1.0, 2.0],
                   [3.0, 4.0]]])
print(corr2d_multi_in(X, K))


tensor([[ 56.,  72.],
        [104., 120.]])


## 多输出通道

每个输出通道都有一组完整的多输入通道卷积核。


In [4]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)


K_multi = torch.stack((K, K + 1, K + 2), 0)
print(K_multi.shape)
print(corr2d_multi_in_out(X, K_multi))


torch.Size([3, 2, 2, 2])
tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])


## 1 x 1 卷积层

1 x 1 卷积不混合空间邻域，但会在每个像素位置混合通道信息，作用类似逐像素的全连接层。


In [5]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))


X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6
print(Y1.shape)


torch.Size([2, 3, 3])


多通道卷积让网络可以在同一层学习多种特征；1 x 1 卷积常用于调整通道数和组合通道信息。
